# 🚀 Phase 2 (M3): Train Qwen2.5-7B-Instruct QLoRA GRPO trên A100 80GB (32B 4-bit Local Judge + Python Constraints)

Huấn luyện Writing Agent **`Qwen/Qwen2.5-7B-Instruct` (QLoRA `r=32`)** trên bộ **640 chapter prompts** (`data/grpo_train_condition_d.jsonl`) với **0 phí API**:
- **VRAM Budget trên 1× A100 80GB:** `~32 GB` (7B Policy + GRPO grads, `G=4`) + `~19.5 GB` (`unsloth/Qwen2.5-32B-Instruct-bnb-4bit` Judge frozen) + `~0.1 GB` (`MiniLM`) = **`~52–58 GB / 80 GB`**.
- **Chuyển đổi cấu hình thí nghiệm chỉ bằng 1 dòng `REWARD_CONFIG`:**
  - `rewards/configs/exp00_baseline.yaml` — EXP-00 (Baseline Phase 1)
  - `rewards/configs/exp01_holistic.yaml` — EXP-01 (Holistic 32B Judge)
  - `rewards/configs/exp02_hierarchical.yaml` — **EXP-02 (Main: `0.65 R_G + 0.35 R_L - P`)**
  - `rewards/configs/exp03_local_only.yaml` — EXP-03 (Local Expansion Only)
  - `rewards/configs/exp04_global_only.yaml` — EXP-04 (Global Chapter Only)
  - `rewards/configs/exp05_hierarchical_geo.yaml` — EXP-05 (Geometric Aggregation)
- Tự động lưu checkpoint & log chi tiết từng trục reward (`phase2_reward_logs.jsonl`) vào Google Drive + hỗ trợ auto-resume.

In [ ]:
# 1. Check A100 80GB, Mount Drive & Sync Repo
!nvidia-smi
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/MA-GRPO')
REPO_URL = 'github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN') or os.environ.get('GH_TOKEN', '')
except Exception:
    GH_TOKEN = os.environ.get('GH_TOKEN', '')

clone_url = f'https://{GH_TOKEN}@{REPO_URL}' if GH_TOKEN else f'https://{REPO_URL}'
if not REPO_DIR.exists():
    !git clone {clone_url} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 2. Install Training Dependencies
!pip install -q trl peft bitsandbytes transformers datasets accelerate sentence-transformers pyyaml tensorboard matplotlib pandas

In [ ]:
# 3. Configure Experiment (EXP-02-RLMR Minimal Orthogonal Default, gamma=0.05, checkpoint every 10 steps)
EXP_NAME = 'exp02_rlmr_orthogonal_32b'
REWARD_CONFIG = 'rewards/configs/exp02_rlmr_orthogonal.yaml'
TRAIN_CONFIG = 'configs/colab_a100.yaml'
# Tự ưu tiên file trên Drive nếu có, không thì dùng bản trong repo
DRIVE_DATA = Path('/content/drive/MyDrive/MA-GRPO-Dataset/grpo_train_condition_d.jsonl')
DATASET_PATH = str(DRIVE_DATA if DRIVE_DATA.exists() else Path('data/grpo_train_condition_d.jsonl'))
OUTPUT_DIR = f'/content/drive/MyDrive/MA-GRPO-Checkpoints/{EXP_NAME}'

MAX_STEPS = 160            # 160 steps x 1 prompt/step (G=4 rollouts = 640 rollouts); full 1 epoch = 640 steps
GROUP_SIZE = 4
BATCH_SIZE = 4             # Batched G=4 trên 1 kernel (thay vì bs=1, ga=4 tuần tự)
GRAD_ACCUM = 1
MAX_COMPLETION_TOKENS = 1950
POLICY_BF16 = True         # Native bf16 LoRA cho 7B trên A100 80GB (~45-52 GB/80 GB, nhanh hơn ~2.5x so với 4-bit!)
SAVE_STEPS = 10            # Checkpoint tự động lên Drive mỗi 10 steps + auto-resume
AUTO_DISCONNECT = False

# Auto-detect latest checkpoint for seamless resume
out_p = Path(OUTPUT_DIR)
out_p.mkdir(parents=True, exist_ok=True)
ckpts = sorted([d for d in out_p.glob('checkpoint-*') if d.is_dir()], key=lambda d: int(d.name.split('-')[-1]))
RESUME_CKPT = str(ckpts[-1]) if ckpts else None
print(f'Dataset: {DATASET_PATH} | Output: {OUTPUT_DIR} | Resume: {RESUME_CKPT}')

In [ ]:
# 4. Launch LoRA GRPO Training with Phase 2 Minimal Orthogonal RLMR (gamma=0.05)
import subprocess
import sys

cmd = [
    sys.executable, '-u', 'scripts/train.py',
    '--dataset', DATASET_PATH,
    '--config', TRAIN_CONFIG,
    '--reward-config', REWARD_CONFIG,
    '--output-dir', OUTPUT_DIR,
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation', str(GRAD_ACCUM),
    '--group-size', str(GROUP_SIZE),
    '--max-completion-tokens', str(MAX_COMPLETION_TOKENS),
    '--save-steps', str(SAVE_STEPS),
    '--reward-device', 'cuda:0',
]
if POLICY_BF16:
    cmd.append('--policy-bf16')
if RESUME_CKPT:
    cmd.extend(['--resume-from-checkpoint', RESUME_CKPT])

print('Running:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'train.py exited with code {ret}')

In [ ]:
# 5. Plot Phase 2 Reward Curves (R_total, R_local, R_global, Penalties & Think/Prose Words)
import json
import pandas as pd
import matplotlib.pyplot as plt

log_file = Path(OUTPUT_DIR) / 'phase2_reward_logs.jsonl'
if log_file.exists():
    df = pd.DataFrame([json.loads(l) for l in log_file.read_text(encoding='utf-8').splitlines() if l.strip()])
    print(f'Logged {len(df)} rollout evaluations.')
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    df[['r_total', 'r_local', 'r_global']].rolling(16, min_periods=1).mean().plot(ax=axes[0], title='Reward Progression')
    df[['p_length', 'p_rep', 'p_contra', 'p_format']].rolling(16, min_periods=1).mean().plot(ax=axes[1], title='Constraint Penalties')
    df[['prose_words', 'think_words']].rolling(16, min_periods=1).mean().plot(ax=axes[2], title='Prose vs Think Words')
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / 'phase2_training_curves.png', dpi=150)
    plt.show()

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()